In [1]:
# split = 'dev'
split = 'train'

mode = 'cevi'
# mode = 'cev'

ordinal=True
# ordinal=False

In [2]:
import json
def read_json_object(file_path):
    with open(file_path, 'r', encoding="utf-8") as file:
        json_data = json.load(file)
    return json_data

In [3]:
data = read_json_object(f"/home/comp/24483737/fc/data/averitec/{split}.json")

In [4]:
from collections import Counter

label_counter = Counter()

for d in data:
    label = d.get('label')
    if label is not None:
        label_counter[label] += 1
    else:
        print("Warning: 某个字典中没有 'label' 字段，已跳过")

for label, count in label_counter.items():
    print(f"{label}: {count}")

Supported: 849
Refuted: 1742
Conflicting Evidence/Cherrypicking: 195
Not Enough Evidence: 282


In [5]:
from collections import Counter

label_counter = Counter()

for d in data:
    ct = d.get('claim_types')
    if ct is not None:
        for c in ct:
            label_counter[c] += 1
    else:
        print("Warning: 某个字典中没有 'label' 字段，已跳过")

for label, count in label_counter.items():
    print(f"{label}: {count}")

Position Statement: 238
Numerical Claim: 1035
Event/Property Claim: 1772
Quote Verification: 295
Causal Claim: 353


In [6]:
len(data)

3068

In [7]:
3068-2873

195

In [8]:
import uuid
def add_deterministic_ids(raw_list):
    for idx, sample in enumerate(raw_list):
        name = f"sample_{idx}" 
        sample['id'] = str(uuid.uuid5(uuid.NAMESPACE_DNS, name))
    return raw_list

In [9]:
aft = add_deterministic_ids(data)

In [10]:
ids = [sample['id'] for sample in aft]
print("是否有重复ID：", len(ids) != len(set(ids)))

是否有重复ID： False


In [13]:
def clean(raw_list, mode, ordinal):
    conv_list = []
    for event in raw_list:
        conv = {}
        conv['id'] = event['id']
        
        claim = event['claim']
        justification = event['justification']
        raw_label = event['label']
        if raw_label == "Conflicting Evidence/Cherrypicking":
            continue

        q_list = event['questions']
        if ordinal:
            evidence_id = 1
        evidences = ''
        
        for q_dict in q_list:
            q = q_dict['question']
            after_q = q if q.endswith('?') else (q + '?')
            
            a_list = q_dict['answers']
            a_str = ''
            for a_dict in a_list:
                a = a_dict['answer']
                after_a = a if a.endswith('.') else (a + '.')
                a_str += after_a + ' '  

            if ordinal:
                evidences += f" ({evidence_id}) {after_q} {a_str}"
                evidence_id += 1
            else:
                evidences += f"{after_q} {a_str}"

        if len(evidences) == 0:
            continue
        
        utter_list = []
        if mode == 'cevi':
            utter_list.append({"from": "human", "value": f"Claim: {claim}\nEvidences: {evidences}"})
            utter_list.append({"from": "gpt", "value": f"Verdict: {raw_label}. Explanation: {justification}"})
        elif mode == 'cev':
            utter_list.append({"from": "human", "value": f"Claim: {claim}\nEvidences: {evidences}"})
            utter_list.append({"from": "gpt", "value": f"Verdict: {raw_label}"})
        else:
            utter_list.append({"from": "human", "value": f"Claim: {claim}"})
            utter_list.append({"from": "gpt", "value": f"Verdict: {raw_label}"})
        conv['conversations'] = utter_list
        conv_list.append(conv)
    return conv_list
   

In [14]:
data = clean(aft, mode, ordinal)
len(data)

2873

In [15]:
data[1]

{'id': 'f5dc1eab-0fad-5229-8e61-bca540e9a713',
 'conversations': [{'from': 'human',
   'value': 'Claim: Donald Trump delivered the largest tax cuts in American history.\nEvidences:  (1) Did the 2017 tax bill deliver the largest tax cuts in American history? This tax cut is the 8th largest as a percent of Gross Domestic Product (GDP) since 1918 and the 4th largest in inflation-adjusted dollars.  (2) Has there been larger tax bills than the 2017 tax bill? Three tax bills have been larger: American Taxpayer Relief Act of 2012 (enacted in 2013); \nTax Relief, Unemployment Insurance Reauthorization, and Job Creation Act of 2010 and\nEconomic Recovery Tax Act of 1981. '},
  {'from': 'gpt',
   'value': 'Verdict: Refuted. Explanation: Three tax bills have been larger than that of Donald Trump'}]}

In [16]:
len(data)

2873

In [17]:
filter_ambi = [] 
filtered = []
q_n = 0
a_n = 0
for sample in data:
    q = sample['conversations'][0]['value']
    a = sample['conversations'][1]['value']
    if ("Refuted") in a and ('no evidence') in a: 
        filtered.append(sample)
        continue
    if ("Supported") in a and ('no evidence') in a:
        filtered.append(sample)
        continue
    filter_ambi.append(sample)

In [18]:
print(len(filter_ambi))
print(q_n)
print(a_n)

2802
0
0


In [133]:
filter_ambi[0]

{'id': '3adbce3b-4554-57d8-ae26-d643865cef5a',
 'conversations': [{'from': 'human',
   'value': 'Claim: Hunter Biden had no experience in Ukraine or in the energy sector when he joined the board of Burisma.\nEvidences:  (1) Did Hunter Biden have any experience in the energy sector at the time he joined the board of the  Burisma energy company in 2014? No.  (2) Did Hunter Biden have any experience in Ukraine at the time he joined the board of the  Burisma energy company in 2014? No. '},
  {'from': 'gpt', 'value': 'Verdict: Supported'}]}

In [136]:
len(data)

2873

In [20]:
if ordinal:
    with open(f'./post-averitec/final_2802_nolabelerror/filtered_{split}_averitec_{mode}_ordinal.json', 'w', encoding="utf-8" ) as file:
        json.dump(filtered, file, indent=2, ensure_ascii=False)
else:
    with open(f'./post-averitec/final_2802_nolabelerror/{split}_averitec_{mode}.json', 'w', encoding="utf-8" ) as file:
        json.dump(filter_ambi, file, indent=2, ensure_ascii=False)